# 4-1회차 | 분류 ① 로지스틱 회귀

**핵심 질문**: "확률로 분류"한다는 건 무슨 뜻인가?

> 3-2회차를 이렇게 닫았음.
> *"숫자가 나왔다는 이유만으로 믿지 않음. 무엇을 맞혔고, 무엇을 놓쳤고, 어떤 실수를 감수했는지 확인함."*
> *"모델을 믿기 전에, 우리가 만든 실험을 먼저 의심해야 함."*
>
> **4회차는 여기서 시작함.** 이번엔 모델을 열어볼 거임 —
> 그 확률이 **어디서 나온 숫자인지**를 확인함.

**오늘의 목표**
1. **분류란?**: 데이터를 보고 어느 그룹에 속하는지 라벨을 붙이는 작업
2. **Sigmoid 함수**: 모델 점수(z)를 확률(0~1)로 바꿔주는 변환기
3. **로지스틱 회귀**: Sigmoid를 이용한 확률 기반 분류기
4. **임곗값(Threshold)**: 임곗값에 따라 Precision/Recall이 줄다리기

**예제**: Titanic 생존 예측 — 예측 확률 출력 → 임곗값 실험

---

> **실행 전 준비물**
> - `train.csv` (Kaggle Titanic) — 이 노트북과 **같은 폴더**에 둘 것
> - 슬라이드 PNG 4장 (`스크린샷 2026-02-16 오후 2.12.24.png` 등) — 없으면 markdown 이미지가 깨짐. 코드 실행에는 지장 없음

---
## 4회차에서 분류를 다루는 이유

**3회차에서 전처리와 평가지표를 배움.**
데이터를 준비하는 방법과 모델 성능을 제대로 평가하는 방법을 알았을 것

이제 본격적으로 **분류 모델**을 배울 차례

### 분류(Classification)란?

데이터를 보고 **어느 그룹에 속하는지 라벨을 붙이는 작업**

| 분야 | 분류 예시 |
|------|----------|
| 금융 | 사기 거래인가, 정상 거래인가? |
| 이메일 | 스팸인가, 정상 메일인가? |
| 의료 | 암 환자인가, 건강한 사람인가? |
| 이미지 | 고양이인가, 강아지인가? |
| 마케팅 | 이 고객이 이탈할 것인가? |

> **4회차 핵심 문장:**  
> "분류 모델은 각각 다른 방식으로 경계를 긋는다 — 확률, 거리, 규칙."

### 대표적인 분류 알고리즘

| 알고리즘 | 핵심 아이디어 | 기준 | 이번 회차 |
|---------|-------------|--------|----------|
| **로지스틱 회귀** | 점수 → 확률 → 분류 | 확률 | **4-1 (지금)** |
| **KNN** | 가까운 이웃끼리 다수결 | 거리 | 4-2 (다음) |

---
## Part 1. Sigmoid 함수 — 점수를 확률로 바꿔주는 변환기

### 왜 확률이 필요한가?

- 모델이 내부적으로 계산한 점수(z)는 **-∞ ~ +∞** 범위
- 하지만 우리가 원하는 건 **"이 사람이 생존할 확률이 몇 %인가?"**
- Sigmoid 함수가 이 점수를 **0 ~ 1 사이의 확률**로 바꿔줌

### Sigmoid 공식

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

| z 값 | sigmoid(z) | 해석 |
|------|-----------|------|
| -10 | ≈ 0.00 | 거의 확실히 음성 |
| 0 | 0.50 | 반반 (애매) |
| +10 | ≈ 1.00 | 거의 확실히 양성 |

> 마치 **시험 점수(z)**를 **합격 확률**로 바꿔주는 변환기!

In [1]:
import numpy as np
import plotly.graph_objects as go

z = np.linspace(-10, 10, 200)

sigmoid = 1 / (1 + np.exp(-z))

fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=sigmoid, mode="lines",
                         line=dict(color="purple", width=3),
                         name="Sigmoid"))

fig.add_hline(y=0.5, line_dash="dash", line_color="grey",
              annotation_text="확률 0.5 (기준선)")
fig.add_vline(x=0, line_dash="dash", line_color="grey")

fig.update_layout(title="Sigmoid Function (점수 → 확률 변환기)",
                  xaxis_title="z (모델 점수)",
                  yaxis_title="sigmoid(z) = 확률",
                  template="plotly_dark",
                  yaxis=dict(range=[-0.05, 1.05]))
fig.show()

### Sigmoid 그래프 해석

- **z가 크면** → sigmoid ≈ 1 → "양성(생존)일 확률이 높다" (확신)
- **z가 작으면** → sigmoid ≈ 0 → "음성(사망)일 확률이 높다" (확신)
- **z = 0** → sigmoid = 0.5 → "반반, 가장 애매한 지점" (**결정 경계**)

> 핵심 1: **z = 0 ⇔ 확률 0.5** 가 기본 기준선
> 핵심 2: 경계 근처는 **민감(확률이 크게 흔들림)**, 양끝은 **둔감(0/1에 붙음)** → 헷갈림/확신이 생김

---
## Part 2. Titanic 데이터 전처리

### Titanic 데이터셋
- 실제 타이타닉호 생존자 데이터
- 목표: 승객의 나이, 성별, 선실 등급 등을 보고 **생존 여부(0=사망, 1=생존)** 예측
- 주요 컬럼:
  - `Pclass`: 선실 등급 (1=일등석, 2=이등석, 3=삼등석)
  - `Sex`: 성별
  - `Age`: 나이
  - `Fare`: 운임 요금
- **Target(정답)**: `Survived` (1=생존, 0=사망)

> 3회차에서 배운 전처리(인코딩, 결측치 처리)를 활용

In [2]:
import pandas as pd
from pathlib import Path

CSV = "train.csv"
if not Path(CSV).exists():
    raise FileNotFoundError(
        f"'{CSV}' 를 이 노트북과 같은 폴더에 두고 다시 실행하셈. "
        "(Kaggle Titanic train.csv)"
    )

titanic_df = pd.read_csv(CSV)
display(titanic_df.head())


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

y_titanic_df = titanic_df["Survived"]
X_titanic_df = titanic_df.drop("Survived", axis=1).copy()

X_titanic_df["Cabin1"] = X_titanic_df["Cabin"].fillna("N").astype(str).str[:1]

X_titanic_df = X_titanic_df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

print(f"데이터 크기: {X_titanic_df.shape}")
print(f"생존 비율:\n{y_titanic_df.value_counts(normalize=True)}")
display(X_titanic_df.head())

# 1단계: 최종 시험지(test)를 먼저 떼어내고 봉인함
X_dev, X_final_test, y_dev, y_final_test = train_test_split(
    X_titanic_df, y_titanic_df,
    test_size=0.2, stratify=y_titanic_df, random_state=42
)

# 2단계: 남은 데이터를 다시 교과서(train) / 모의고사(valid)로 나눔
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev,
    test_size=0.25, stratify=y_dev, random_state=42
)

print(f"train   (교과서)   : {len(X_train)}명")
print(f"valid   (모의고사) : {len(X_valid)}명")
print(f"test (수능)    : {len(X_final_test)}명  ← 맨 마지막에 딱 한 번만 씀")

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked", "Cabin1"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

lr = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=5000, random_state=42)),
])

데이터 크기: (891, 8)
생존 비율:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin1
0,3,male,22.0,1,0,7.2500,S,N
1,1,female,38.0,1,0,71.2833,C,C
2,3,female,26.0,0,0,7.9250,S,N
3,1,female,35.0,1,0,53.1000,S,C
4,3,male,35.0,0,0,8.0500,S,N


train   (교과서)   : 534명
valid   (모의고사) : 178명
test (수능)    : 179명  ← 맨 마지막에 딱 한 번만 씀


### 3-2회차와 같은 3분할 — 새로 배우는 게 아님

3-2회차 Part 5에서 threshold를 고를 때 이미 이 구조를 썼음.

> 배울 때는 **train**만 봄.
> 선택할 때는 **validation**을 봄.
> **test**는 모든 선택이 끝난 뒤 마지막에 한 번 봄.

**오늘 새로 만든 규칙이 아님.** 3-2에서는 digits 데이터로 했고, 오늘은 Titanic으로 함.
**데이터와 모델이 바뀌어도 절차는 그대로**라는 걸 확인하는 게 요점임.

---

#### 오늘 valid에서 고를 것들

| 무엇 | 3-2에서는 | 4-1에서는 |
|---|---|---|
| 임곗값(threshold) | digits, PCA+LR | **Titanic, 전체 피처 LR** |
| 피처 조합 | — | **오늘 추가됨** |
| 인코딩 방식 | — | **오늘 추가됨** |

3-2보다 **고를 것이 늘었음.** 고를 게 늘어날수록 test를 지키는 게 더 중요해짐.

> **오늘의 규칙**: 노트북이 끝날 때까지 `X_final_test`는 **한 번도 안 나옴.**
> 맨 마지막 셀에서 딱 한 번 등장함.

---
## Part 3. 로지스틱 회귀로 Titanic 생존 예측

### 로지스틱 회귀란?

- "회귀"라는 이름이 붙어있지만 **분류 모델**임
- 내부적으로 **선형 회귀처럼 점수(z)를 계산**한 뒤
- **Sigmoid 함수로 확률(0~1)로 변환**
- 확률이 임곗값(기본 0.5) 이상이면 → 양성(1)

```
입력 데이터 → 점수(z) 계산 → Sigmoid → 확률 → 임곗값 비교 → 예측
```

| 단계 | 내용 | 예시 |
|------|------|------|
| 1 | 점수 계산 | z = 2.3 |
| 2 | Sigmoid 변환 | σ(2.3) = 0.909 |
| 3 | 임곗값 비교 | 0.909 ≥ 0.5 → 양성(생존) |

### 로지스틱 회귀의 본질

로지스틱 회귀를 한 마디로 말하면:

> **두 클래스를 가르는 직선을 긋고, 그 선에서 얼마나 떨어졌는지(= 점수 z)를 sigmoid로 확률로 바꾼 뒤, 임곗값으로 분류하는 모델**

> (핵심: 로지스틱은 0/1을 바로 찍지 않고, 확률을 먼저 만든다)

---

#### 경계선(Decision Boundary)이란?

![로지스틱 회귀는 데이터를 가르는 직선을 긋는 모델](스크린샷%202026-02-16%20오후%202.13.03.png)

로지스틱 회귀는 두 클래스를 나누는 **직선(경계선)**을 찾음.


| 위치 | 확률 | 의미 |
|------|------|------|
| 직선 위 (경계선) | 0.5 | 반반, 애매함 |
| 직선에서 **양성 쪽**으로 멀어짐 | → 1.0 | 양성일 가능성 높음 |
| 직선에서 **음성 쪽**으로 멀어짐 | → 0.0 | 음성일 가능성 높음 |

> **주의**: 직선은 "예측값"이 아니라, **두 클래스를 나누는 기준선**임!  
> 경계선에 가까울수록 확률이 0.5에 가깝고 (모델이 헷갈림),  
> 멀어질수록 확률이 0이나 1에 가깝습니다 (모델이 확신함).

![선과의 거리가 곧 확신(Probability)이 됩니다](스크린샷%202026-02-16%20오후%202.13.29.png)

> **"선과의 거리"와 "점수 z"의 관계**
>
> 정확히는 **z = (선까지의 거리) × ‖w‖** 임. 같은 값이 아니라 **비례** 관계임.
> 선 위에 있으면 거리 = 0 → z = 0 → 확률 0.5.
> 선에서 멀어질수록 z가 비례해서 커지고, 확률이 0이나 1 쪽으로 밀림.
>
> **순서는 똑같고 배율만 다름.** 그래서 "멀수록 확신이 커진다"는 말은 맞음.
> 이따가 이 관계를 숫자로 직접 확인함.

---

#### 그 직선이 "잘 나눈다"는 걸 어떻게 아는가?

1회차에서 배운 **손실함수**를 기억!  
로지스틱 회귀는 **Cross Entropy Loss**를 사용함:

$$\text{loss} = -\big[\,y\log \hat p + (1-y)\log(1-\hat p)\,\big] \;+\; \text{규제}$$

| 상황 | 벌점 |
|------|------|
| 정답 클래스의 확률이 **1에 가까우면** | 벌점 아주 작음 |
| 정답 클래스의 확률이 **0.5면** | 벌점 상당함 |
| 정답 클래스의 확률이 **0에 가까우면** | 벌점 **엄청 큼!** |

> **뒤에 붙은 "규제"가 뭐임?**
> `LogisticRegression()`은 기본적으로 **L2 규제**가 켜져 있음.
> 계수가 너무 커지지 않게 누르는 항임 (`C` 파라미터로 강도 조절).
> 그래서 모델은 "벌점만" 줄이는 게 아니라 **"벌점 + 계수 크기"**를 같이 줄임.

> 1회차에서 본 "메인 재료 빠지면 극대노 미식가" = Cross Entropy!

---

#### 학습 과정 (5단계)

| 단계 | 모델이 하는 일 |
|------|--------------|
| 1 | 직선을 **아무렇게나** 긋는다 |
| 2 | 각 데이터의 **확률을 계산**한다 |
| 3 | 정답과 비교해서 **벌점(Cross Entropy)**을 받는다 |
| 4 | 벌점이 줄어드는 방향으로 **직선을 조금 움직인다** |
| 5 | 이걸 **손실이 더 줄지 않을 때까지 반복** → 최적 직선에 도착! |

![벌점을 줄이기 위해 끊임없이 선을 움직입니다](스크린샷%202026-02-16%20오후%202.14.23.png)

> **핵심**: 직접 "잘 나누는지" 판단하는 게 아님!  
> **확률 예측 오차(Cross Entropy)를 최소화하다 보니**,  
> 결과적으로 잘 나누는 선이 생기는 것!

> **정확히 말하면**: 모델이 선을 직접 잡고 움직이는 게 아님.
> **`w`와 `b`를 조금씩 바꾸고, 그 결과로 선의 위치가 바뀜.**
>
> 그리고 위 그림은 **경사하강법(Gradient Descent)** 의 그림인데,
> `LogisticRegression()`의 기본 최적화기는 경사하강법이 아니라 **`lbfgs`** 임.
> 아이디어(손실이 줄어드는 방향으로 간다)는 같지만 방법은 다름.
> 실제로 몇 번 돌았는지는 아래에서 직접 확인함.

---

#### 로지스틱 회귀가 하는 일 (한 문장 정리)

> **정답 클래스는 확률이 1에 가깝게,  
> 반대 클래스는 확률이 0에 가깝게**  
> 만드는 직선을 찾아가는 것

---

#### 로지스틱 회귀의 가정 — "세상이 직선이 아니면?"

로지스틱 회귀는 **"전처리된 피처에 대해 log-odds가 선형"이라고 가정**함.

쉽게 말하면 **경계선을 직선(고차원에서는 평면)으로 긋는다**는 뜻임.
데이터가 완벽히 직선으로 갈라져야 한다는 뜻은 **아님** — 겹쳐도 잘 돌아감.
다만 경계 자체가 휘어야 하는 문제라면 약해짐

| 상황 | 결과 |
|------|------|
| 직선으로 나눌 수 있는 데이터 | 잘 작동함 |
| 직선으로 나눌 수 **없는** 데이터 | 한계가 있음 |

> 세상이 직선이 아니면?  
> → **모델을 바꾸거나** (결정 트리, SVM 등)  
> → **입력을 바꿔야** 합니다 (다항식 특성 추가 등)  
> 이건 다음 시간에 결정 트리를 배우면서 비교해볼 예정!

In [4]:
lr.fit(X_train, y_train)

y_proba = lr.predict_proba(X_valid)[:, 1]

display(pd.DataFrame({"예측 확률 (생존)": y_proba[:10].round(4)}))

,예측 확률 (생존)
0,0.1093
1,0.1176
2,0.7400
3,0.1017
4,0.0840
5,0.1017
6,0.0840
7,0.9173
8,0.0842
9,0.5570


In [5]:
lr_model = lr.named_steps["model"]

print(f"solver          : {lr_model.solver}")
print(f"max_iter (설정) : {lr_model.max_iter}")
print(f"n_iter_  (실제) : {lr_model.n_iter_[0]}")


solver          : lbfgs
max_iter (설정) : 5000
n_iter_  (실제) : 32


### `max_iter=5000`인데 실제로는 몇 번 돌았음?

**`max_iter`는 상한이지 실행 횟수가 아님.**
"수렴하지 않으면 여기서 멈춰라"는 안전장치일 뿐임.

- `solver=lbfgs` — 경사하강법이 아니라 **준뉴턴법** 계열임
- 근데 원리는 비슷함.
- 손실이 더 줄지 않으면 **상한과 무관하게 알아서 멈춤**

> `max_iter`를 5000으로 준 이유는 "많이 돌리려고"가 아니라
> **"수렴 경고를 안 보려고"** 임. 성능을 올리는 값이 아님.

> **확인한 만큼만 말한다**: 위 숫자는 이 데이터·이 전처리에서 나온 값임.
> 데이터가 바뀌면 반복 횟수도 바뀜.

### predict vs predict_proba

| 메서드 | 반환값 | 예시 |
|--------|--------|------|
| `predict()` | 0 또는 1 | [0, 1, 0, 1] |
| `predict_proba()` | 각 클래스의 확률 | [[0.92, 0.08], [0.27, 0.73]] |

- `predict_proba()[:, 0]` = 사망 확률
- `predict_proba()[:, 1]` = 생존 확률

> `predict()`는 내부적으로 `predict_proba()` → 임곗값 0.5 비교를 자동으로 해주는 것!

---

### [4-1] Prediction Card 1 — 손실

> **"살짝 틀리기" vs "확신하고 틀리기", 대가가 얼마나 다름?**

방금 Cross Entropy를 배웠음. 그럼 이걸 먼저 예상해보셈.

실제 정답은 **사망(0)** 인데 모델이 이렇게 예측했다면, 벌점은 각각 얼마일까?

| 모델이 낸 생존 확률 | 벌점 예상 |
|---|---|
| 0.51 (거의 반반) | |
| 0.99 (확신) | |

- 0.99로 틀린 게 0.51로 틀린 것보다 **몇 배** 더 아플 것 같음? 2배? 10배?

In [6]:
import numpy as np

for q in [0.51, 0.7, 0.9, 0.99]:
    loss = -np.log(1 - q)
    print(f"정답=사망(0)인데 생존 확률 {q:.2f} 로 예측 → 벌점 {loss:.4f}")

print()
print(f"0.99로 틀린 벌점 / 0.51로 틀린 벌점 = {-np.log(1-0.99) / -np.log(1-0.51):.2f}배")


정답=사망(0)인데 생존 확률 0.51 로 예측 → 벌점 0.7133
정답=사망(0)인데 생존 확률 0.70 로 예측 → 벌점 1.2040
정답=사망(0)인데 생존 확률 0.90 로 예측 → 벌점 2.3026
정답=사망(0)인데 생존 확률 0.99 로 예측 → 벌점 4.6052

0.99로 틀린 벌점 / 0.51로 틀린 벌점 = 6.46배


### 결과 — 확신할수록 대가가 커짐

**6.46배.** 확률을 0.51에서 0.99로 밀었을 뿐인데 벌점은 6배 넘게 뜀.

> 그래서 로지스틱 회귀는 **함부로 극단으로 밀지 않음.**
> 확신했다가 틀리면 손해가 크니까, **애매한 건 애매하게** 두는 게 유리함.

이게 3-2에서 배운 것과 이어짐 — 정확도만 보면 "맞았다/틀렸다"뿐이지만,
Cross Entropy는 **얼마나 확신했는지까지** 채점함.

In [7]:
from sklearn.metrics import accuracy_score

y_pred = lr.predict(X_valid)
print(f"기본 Accuracy (threshold=0.5): {accuracy_score(y_valid, y_pred):.3f}")

기본 Accuracy (threshold=0.5): 0.798


---

### [4-1] Prediction Card 2 — 기준선

`Accuracy 0.798`. 좋은 숫자임? **비교 대상이 없으면 알 수 없음.**

3-2회차 Part 1을 기억하셈. 거기서 이런 걸 만들었음.

```python
# 학습이 아니라 그냥 규칙
"남자면 사망(0), 여자면 생존(1)"
```

그때 이 한 줄짜리 규칙이 **77%** 를 냈고, 우리는 "정확도만 보면 착시"라고 배웠음.

**예상해보셈.**

| | valid Accuracy 예상 |
|---|---|
| ① 전부 사망이라고 찍기 (다수 클래스) | |
| ② 3-2의 Dummy 규칙 (남자=사망) | |
| ③ 우리가 만든 로지스틱 | 0.798 (이미 나옴) |

- ②와 ③의 차이가 얼마나 날 것 같음? 5%p? 10%p?
- 피처 8개를 쓰고 Pipeline까지 붙였으니 꽤 이기지 않을까?

In [8]:
import numpy as np
from sklearn.metrics import roc_auc_score

def dummy_rule(X):
    """3-2회차 Part 1의 규칙: 남자면 사망(0), 아니면 생존(1)"""
    return np.where(X["Sex"].values == "male", 0, 1)

rows = []
for name, Xs, ys in [("valid", X_valid, y_valid)]:
    majority = np.zeros(len(ys), dtype=int)
    rows.append(["① 전부 사망 (다수 클래스)", round(accuracy_score(ys, majority), 4), "-"])
    rows.append(["② 3-2 Dummy 규칙 (남자=사망)", round(accuracy_score(ys, dummy_rule(Xs)), 4), "확률 없음"])
    rows.append(["③ 로지스틱 회귀 (피처 8개)",
                 round(accuracy_score(ys, (y_proba >= 0.5).astype(int)), 4),
                 round(roc_auc_score(ys, y_proba), 4)])

display(pd.DataFrame(rows, columns=["모델", "valid Accuracy", "AUC"]))


,모델,valid Accuracy,AUC
0,① 전부 사망 (다수 클래스),0.6180,-
1,② 3-2 Dummy 규칙 (남자=사망),0.7978,확률 없음
2,③ 로지스틱 회귀 (피처 8개),0.7978,0.857


### [4-1] Prediction Card 2 Reveal — 충격

**②와 ③이 같음.**

피처 8개, OneHot 인코딩 19차원, Pipeline, `max_iter=5000`.
그렇게 만든 모델이 **`if 남자: 사망` 한 줄과 정확도가 똑같음.**

> 3-2에서 "이건 학습이 아니라 그냥 규칙임"이라고 했던 그 규칙임.

---

#### 그럼 로지스틱을 왜 배움?

**AUC 열을 보셈.** Dummy는 **비어 있음.**

| | Accuracy | 확률 출력 | AUC |
|---|---|---|---|
| Dummy 규칙 | 같음 | **불가능** | **잴 수 없음** |
| 로지스틱 | 같음 | `predict_proba` | **0.857** |

Dummy는 0 아니면 1만 뱉음. **"얼마나 확실한가"를 물어볼 수 없음.**
그래서:

- **임곗값을 바꿀 수 없음** — 조정할 확률이 없으니까
- **순위를 매길 수 없음** — 누가 더 위험한지 못 고름
- **AUC를 잴 수 없음** — 순위가 없으니까

> **오늘의 문장**:
> 로지스틱이 Dummy보다 나은 건 **더 많이 맞혀서가 아니라, 확률을 주기 때문**임.
> 그 확률이 있어야 임곗값을 정책에 맞게 고를 수 있음.

---

#### 3-2 회수

3-2회차에서 이렇게 배웠음.

> Accuracy는 출발점일 수 있지만, **한 숫자만으로 모델을 믿지는 않음.**

방금 그 상황을 정면으로 만났음. **Accuracy로는 두 모델을 구별할 수 없었고, AUC로는 구별됐음.**

> **확인한 만큼만 말한다**: Accuracy가 같다고 두 모델이 같은 건 아님.
> 반대로 AUC가 높다고 반드시 좋은 것도 아님 — **무엇을 위해 쓸 모델인지**에 달렸음.

---
## Part 4. 임곗값(Threshold)에 따른 Precision/Recall 줄다리기

### 3-2 회수 — 지표와 혼동행렬

Precision / Recall / F1 / 혼동행렬은 **3-2회차에서 이미 다 함.**
정의를 다시 설명하지 않음. 기억 안 나면 3-2 노트북 Part 2~3을 열어보셈.

> Precision = TP / (TP + FP) · Recall = TP / (TP + FN)

**오늘 다른 점**: 3-2는 digits(7 vs 나머지)였고, 오늘은 **Titanic 생존**임.
같은 지표가 **다른 문제에서도 같은 방식으로 움직이는지** 확인하는 게 목적임.

### 임곗값을 바꾸면?

| 임곗값 | Recall | Precision | 의미 |
|--------|--------|-----------|------|
| 낮춤 (0.3) | ↑ | ↓ | 더 많이 양성으로 잡음 (누락↓, 오탐↑) |
| 높임 (0.7) | ↓ | ↑ | 확실할 때만 양성 (오탐↓, 누락↑) |

> Precision과 Recall은 대체로 **줄다리기** 관계임.
> 임곗값을 올리면 보통 Precision↑ / Recall↓ 방향으로 움직임.

> **단, "항상" 그런 건 아님.** 이따가 표에서 예외를 직접 찾아보셈.


> 그래서 이 문제(타이타닉)에서 FP와 FN 중 무엇이 더 비싼가?


In [9]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import pandas as pd

for thr in [0.3, 0.5, 0.7]:
    y_hat = (y_proba >= thr).astype(int)

    print(f"Threshold={thr}")
    print(f"  Precision: {precision_score(y_valid, y_hat, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_valid, y_hat, zero_division=0):.4f}")

    cm = confusion_matrix(y_valid, y_hat)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0 (사망)", "Actual 1 (생존)"],
        columns=["Pred 0 (사망)", "Pred 1 (생존)"]
    )
    display(cm_df)

    tn, fp, fn, tp = cm.ravel()
    print(f"  FP(오탐): {fp}   FN(누락): {fn}   TP: {tp}   TN: {tn}")
    print("-" * 40)


Threshold=0.3
  Precision: 0.6706
  Recall   : 0.8382


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),82,28
Actual 1 (생존),11,57


  FP(오탐): 28   FN(누락): 11   TP: 57   TN: 82
----------------------------------------
Threshold=0.5
  Precision: 0.7286
  Recall   : 0.7500


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),91,19
Actual 1 (생존),17,51


  FP(오탐): 19   FN(누락): 17   TP: 51   TN: 91
----------------------------------------
Threshold=0.7
  Precision: 0.8140
  Recall   : 0.5147


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),102,8
Actual 1 (생존),33,35


  FP(오탐): 8   FN(누락): 33   TP: 35   TN: 102
----------------------------------------


> - 임곗값 0.3 → Recall 높음 (많이 잡음), Precision 낮음 (오탐 많음)
> - 임곗값 0.7 → Precision 높음 (확실할 때만), Recall 낮음 (놓침 많음)
> - 임곗값 0.5 → 중간 균형

### Selection Rule — 표를 보기 전에 정함

3-2회차에서 세운 규칙임. **표를 본 뒤에 기준을 정하면, 기준 자체를 데이터에 맞춘 것이 됨.**
3분할을 해놓고도 규율이 반쯤 무너짐.

그래서 **먼저 선언함.**

> 이번 실습에서도 3-2와 동일하게,
> **validation F1이 가장 높은 threshold**를 선택하겠음.

실제 업무에서 F1 최대가 정답인 것은 아님.
놓치면 안 되는 문제라면 Recall 조건을 먼저 정할 수도 있고,
FP와 FN의 실제 비용을 기준으로 선택할 수도 있음.

---

### [4-1] Prediction Card 3 — 임곗값

> **Titanic에서도 3-2(digits)와 같은 방향으로 움직일까?**

- threshold를 **낮추면** Precision과 Recall 중 무엇이 오르고 무엇이 내려감?
- F1이 최대가 되는 threshold는 대략 어디쯤일 것 같음? (0.3 / 0.5 / 0.7)

예상한 뒤 아래 validation 표를 확인할 것.

In [10]:
from sklearn.metrics import f1_score
import plotly.graph_objects as go

thr_list = np.round(np.arange(0.1, 1.0, 0.1), 2)
rows = []
for thr in thr_list:
    y_hat = (y_proba >= thr).astype(int)
    rows.append([
        thr,
        precision_score(y_valid, y_hat, zero_division=0),
        recall_score(y_valid, y_hat, zero_division=0),
        f1_score(y_valid, y_hat, zero_division=0)
    ])

thr_df = pd.DataFrame(rows, columns=["Threshold", "Precision", "Recall", "F1"])
display(thr_df)

,Threshold,Precision,Recall,F1
0,0.1,0.453901,0.941176,0.612440
1,0.2,0.641304,0.867647,0.737500
2,0.3,0.670588,0.838235,0.745098
3,0.4,0.705128,0.808824,0.753425
4,0.5,0.728571,0.750000,0.739130
5,0.6,0.803571,0.661765,0.725806
6,0.7,0.813953,0.514706,0.630631
7,0.8,0.888889,0.352941,0.505263
8,0.9,0.947368,0.264706,0.413793


In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=thr_df["Threshold"], y=thr_df["Precision"],
                         mode="lines+markers", name="Precision",
                         line=dict(dash="dash", color="cyan")))
fig.add_trace(go.Scatter(x=thr_df["Threshold"], y=thr_df["Recall"],
                         mode="lines+markers", name="Recall",
                         line=dict(color="magenta")))
fig.add_trace(go.Scatter(x=thr_df["Threshold"], y=thr_df["F1"],
                         mode="lines+markers", name="F1",
                         line=dict(color="yellow")))

fig.update_layout(title="임곗값 변화에 따른 Precision / Recall / F1",
                  xaxis_title="Threshold",
                  yaxis_title="Score",
                  template="plotly_dark",
                  yaxis=dict(range=[0, 1]))
fig.show()

### [4-1] Prediction Card 3 Reveal

3-2에서 본 **업무별 지표 우선순위 표**(스팸 / 암 진단 / 금융 사기 / 추천)를 떠올리셈.
그 표가 여기서도 그대로 적용됨 — 다시 쓰지 않음.

> **임곗값은 "정답"이 없음.** 업무 목표(정책)에 따라 선택함.
> 우리는 위에서 **F1 최대**를 선언했으므로 그걸 따름.

**Titanic에서 물어볼 것 하나**:
FP(살 사람을 죽는다고 함) vs FN(죽을 사람을 산다고 함) — 여기선 어느 쪽이 더 비쌈?
3-2의 암 진단·스팸 필터 사례 중 어느 쪽에 가까움?

---

### [4-1] Prediction Card 4 — 줄다리기

> **임곗값을 올리면 Precision은 항상 올라감?**

임곗값을 **올렸는데 Precision이 오히려 내려가는** 구간이 있을까?

- 있을 것 같음? 없을 것 같음?
- 있다면 임곗값이 **높은 쪽**일까 **낮은 쪽**일까?

> 0.05 간격으로 촘촘하게 훑어서 직접 찾아보셈.

In [12]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

fine = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows, drops = [], []
prev = None
for t in fine:
    yh = (y_proba >= t).astype(int)
    pr = precision_score(y_valid, yh, zero_division=0)
    rc = recall_score(y_valid, yh, zero_division=0)
    n_pos = int(yh.sum())
    rows.append([t, round(pr, 4), round(rc, 4), n_pos])
    if prev is not None and pr < prev - 1e-9:
        drops.append((t, prev, pr))
    prev = pr

display(pd.DataFrame(rows, columns=["Threshold", "Precision", "Recall", "양성 예측 수"]))

print()
if drops:
    print(f"임곗값을 올렸는데 Precision이 내려간 구간: {len(drops)}곳")
    for t, before, after in drops:
        print(f"  thr={t}  Precision {before:.4f} → {after:.4f}")
else:
    print("이 구간에서는 Precision이 단조 증가함")


,Threshold,Precision,Recall,양성 예측 수
0,0.05,0.4024,1.0000,169
1,0.10,0.4539,0.9412,141
2,0.15,0.5882,0.8824,102
3,0.20,0.6413,0.8676,92
4,0.25,0.6477,0.8382,88
5,0.30,0.6706,0.8382,85
6,0.35,0.6707,0.8088,82
7,0.40,0.7051,0.8088,78
8,0.45,0.7067,0.7794,75
9,0.50,0.7286,0.7500,70



임곗값을 올렸는데 Precision이 내려간 구간: 2곳
  thr=0.65  Precision 0.8036 → 0.7708
  thr=0.95  Precision 0.9474 → 0.9167


### 결과 — 줄다리기는 "경향"이지 "보장"이 아님

임곗값을 올렸는데 **Precision이 오히려 내려간 구간이 존재함.**

**왜 그럼?** 오른쪽 `양성 예측 수` 열을 보셈.
임곗값이 높아질수록 양성으로 찍는 사람이 **급격히 줄어듦.**
몇 명 안 남은 상태에서는 **한 명만 틀려도 비율이 크게 흔들림.**

> **정리**
> Precision/Recall 트레이드오프는 **일반적 경향**이지 수학 법칙이 아님.
> 표본이 얇아지는 구간에서는 숫자가 요동침.

> **그래서 하면 안 되는 말**: "임곗값을 올리면 Precision은 반드시 올라갑니다"
> **해야 하는 말**: "보통은 올라가지만, 양성 예측이 몇 개 안 남으면 흔들립니다"

> **확인한 만큼만 말한다**: 위 결과는 valid 178명 기준임.
> 데이터가 커지면 이 흔들림은 줄어듦. **하락 구간의 위치도 split마다 달라짐.**

---
## Part 5. 확률 분포 시각화

모델이 출력한 **확률 분포**를 보면 임곗값 선택의 감이 옴

In [13]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=y_proba[y_valid == 0], name="사망 (실제=0)",
    opacity=0.6, marker_color="cyan",
    xbins=dict(start=0, end=1, size=0.05)
))
fig.add_trace(go.Histogram(
    x=y_proba[y_valid == 1], name="생존 (실제=1)",
    opacity=0.6, marker_color="magenta",
    xbins=dict(start=0, end=1, size=0.05)
))

fig.add_vline(x=0.5, line_dash="dash", line_color="yellow",
              annotation_text="threshold=0.5")

fig.update_layout(
    title="실제 라벨별 예측 확률 분포",
    xaxis_title="예측 확률 (생존)",
    yaxis_title="빈도",
    barmode="overlay",
    template="plotly_dark"
)
fig.show()

### 확률 분포 해석

- **사망(파란색)** 분포가 왼쪽(낮은 확률)에 몰려 있으면 → 모델이 사망자를 잘 구분
- **생존(분홍색)** 분포가 오른쪽(높은 확률)에 몰려 있으면 → 모델이 생존자를 잘 구분
- 두 분포가 **겹치는 구간** = 모델이 헷갈리는 구간 → 임곗값 조정이 필요한 곳

> 좋은 모델일수록 두 분포가 **확실하게 분리**됨

---

### [4-1] Prediction Card 5 — 확신

> **`predict_proba`가 0.97이면 맞은 거임?**

`predict_proba`가 0.97이면 "97% 확신"이라고 말하고 싶어짐.
그런데 진짜 그럴까?

- 확률이 **0.9 이상인데 실제로는 사망**한 사람이 있을 것 같음? 몇 명쯤?
- 확률이 **0.1 이하인데 실제로는 생존**한 사람은?

> 있다면, "확률 = 확신"이라는 말을 다시 생각해봐야 함.

In [14]:
chk = pd.DataFrame({"예측 확률": y_proba.round(3), "실제": y_valid.values})
chk["예측 라벨"] = (chk["예측 확률"] >= 0.5).astype(int)
wrong = chk[chk["예측 라벨"] != chk["실제"]]

extreme = wrong[(wrong["예측 확률"] >= 0.9) | (wrong["예측 확률"] <= 0.1)]

print(f"전체 valid {len(chk)}명 중 틀린 예측: {len(wrong)}명")
print(f"그중 모델이 '확신'했는데 틀린 경우: {len(extreme)}명")
print()
display(extreme.sort_values("예측 확률"))


전체 valid 178명 중 틀린 예측: 36명
그중 모델이 '확신'했는데 틀린 경우: 5명



,예측 확률,실제,예측 라벨
108,0.075,1,0
54,0.089,1,0
105,0.100,1,0
162,0.100,1,0
43,0.956,0,1


### 결과 — 모델은 확신하면서 틀릴 수 있음

위 표를 보셈. **확률이 0.9를 넘는데 실제로는 사망**한 사람이 있고,
반대로 **확률이 0.1도 안 되는데 살아남은** 사람도 있음.

> **`predict_proba = 0.97`은 "97% 맞다"는 뜻이 아님.**
> **"모델이 생존 쪽으로 강하게 기울었다"**는 뜻임.

---

#### 그럼 확률 숫자를 어떻게 읽어야 함?

| 이렇게 읽으면 ❌ | 이렇게 읽어야 ⭕ |
|---|---|
| "97% 확신한다" | "생존 쪽으로 강하게 판단했다" |
| "확률 0.97이면 97%가 산다" | "그러려면 **보정(calibration)** 확인이 필요함" |

확률값이 실제 빈도와 맞는지 검사하는 걸 **calibration**이라고 함.
오늘은 이름만 알고 가고, **"확률 ≠ 정답 보장"** 만 기억하셈.

> **확인한 만큼만 말한다**: 위 결과는 valid 179명 기준임.
> 표본이 작아서 "몇 명"이라는 숫자 자체를 일반화하면 안 됨.

### [4-1] Prediction Card 6 — 분포

> **피처를 늘리면 확률 분포가 어떻게 변함?**

아래에서 피처를 늘려가며 3개 모델의 확률 분포를 그릴 거임.
**실행 버튼 누르기 전에** 각자 예측부터 적어보셈.

| 모델 | 사용 피처 | 두 분포가 겹칠까, 갈라질까? | AUC 예상 (0.5=랜덤, 1.0=완벽) |
|------|----------|--------------------------|---------------------------|
| ① | Pclass만 | | |
| ② | Pclass + Sex | | |
| ③ | 전체 피처 | | |

같이 생각해볼 것:
- **①→② 변화**와 **②→③ 변화** 중 어느 쪽이 더 클 것 같음?
- 피처를 8개까지 늘리면 AUC가 1.0에 가까워질 것 같음?

> 적었으면 실행하셈. **틀려도 됨 — 틀린 예측이 기억에 남음.**

In [15]:
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

all_numeric = {"Pclass", "Age", "SibSp", "Parch", "Fare"}
all_categorical = {"Sex", "Embarked", "Cabin1"}

def make_lr_pipeline(selected_features):
    selected_features = list(selected_features)
    num_feats = [f for f in selected_features if f in all_numeric]
    cat_feats = [f for f in selected_features if f in all_categorical]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_feats),
            ("cat", categorical_transformer, cat_feats),
        ],
        remainder="drop",
    )

    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=5000, random_state=42)),
    ])


configs = [
    ("Pclass만", ["Pclass"]),
    ("Pclass + Sex", ["Pclass", "Sex"]),
    ("전체 피처", X_train.columns.tolist()),
]

titles, probas = [], []
for name, feats in configs:
    m = make_lr_pipeline(feats)
    m.fit(X_train, y_train)
    p = m.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, p)
    titles.append(f"{name} (AUC={auc:.3f})")
    probas.append(p)

fig = make_subplots(rows=3, cols=1, subplot_titles=titles)

for i, p in enumerate(probas, 1):
    fig.add_trace(go.Histogram(
        x=p[y_valid == 0], name="사망 (0)",
        opacity=0.6, marker_color="cyan",
        xbins=dict(start=0, end=1, size=0.05),
        legendgroup="died", showlegend=(i == 1)
    ), row=i, col=1)

    fig.add_trace(go.Histogram(
        x=p[y_valid == 1], name="생존 (1)",
        opacity=0.6, marker_color="magenta",
        xbins=dict(start=0, end=1, size=0.05),
        legendgroup="survived", showlegend=(i == 1)
    ), row=i, col=1)

fig.update_layout(
    height=900,
    barmode="overlay", template="plotly_dark",
    title="모델이 좋아질수록 확률 분포가 분리!",
)

for i in range(1, 4):
    fig.update_xaxes(range=[0, 1], row=i, col=1)

fig.show()

### 해석

| 모델 | 분포 모양 | AUC | 의미 |
|------|----------|-----|------|
| Pclass만 | 거의 겹침 | 낮음 | 구분 능력 부족 |
| Pclass + Sex | 어느 정도 분리 | 중간 | 성별 추가로 크게 개선 |
| 전체 피처 | 확실히 분리 | 높음 | 좋은 분류 모델 |

> **핵심**: 좋은 모델 = 사망자에게 **낮은 확률**, 생존자에게 **높은 확률**을 확신 있게 부여  
> → 두 분포가 **멀리 떨어질수록** 분류 성능이 좋다!  
> → 이것이 바로 **AUC가 높다**는 의미!

---

### 확인 — z는 "선까지의 거리"의 몇 배임?

앞에서 **z = 거리 × ‖w‖** 라고 했음. 진짜인지 숫자로 보셈.

`decision_function()`이 z를 그대로 돌려줌.
그걸 `‖w‖`(계수 벡터의 길이)로 나누면 **실제 기하학적 거리**가 나옴.

In [16]:
import numpy as np

Xt = lr.named_steps["preprocess"].transform(X_valid)
if hasattr(Xt, "toarray"):
    Xt = Xt.toarray()

z = lr_model.decision_function(Xt)
w_norm = np.linalg.norm(lr_model.coef_[0])
dist = z / w_norm

print(f"‖w‖ = {w_norm:.4f}")
print(f"z    범위 : {z.min():.3f} ~ {z.max():.3f}")
print(f"거리  범위 : {dist.min():.3f} ~ {dist.max():.3f}")
print()

display(pd.DataFrame({
    "z": z[:5].round(3),
    "선까지 거리 (z / ‖w‖)": dist[:5].round(3),
    "확률 σ(z)": (1 / (1 + np.exp(-z[:5]))).round(3),
    "실제 라벨": y_valid.values[:5]
}))


‖w‖ = 2.5839
z    범위 : -3.304 ~ 4.086
거리  범위 : -1.279 ~ 1.581



,z,선까지 거리 (z / ‖w‖),확률 σ(z),실제 라벨
0,-2.098,-0.812,0.109,0
1,-2.015,-0.780,0.118,0
2,1.046,0.405,0.740,1
3,-2.178,-0.843,0.102,0
4,-2.389,-0.925,0.084,0


### 해석

- `z`와 `거리`는 **정확히 ‖w‖배 차이**남 — 비례 관계임
- 순위는 똑같음. 그래서 **AUC(순위 기반 지표)는 둘 중 뭘 써도 같음**
- 확률로 바꿀 때 쓰는 건 **거리가 아니라 z** 임 (`σ(z)`)

> 슬라이드의 "선과의 거리가 곧 확신이 된다"는 **방향은 맞음.**
> 다만 sigmoid에 들어가는 건 거리가 아니라 z이고, 그 배율이 `‖w‖`임.
> **‖w‖가 크면 같은 거리라도 확률이 더 극단으로 밀림.**

> **확인한 만큼만 말한다**: `‖w‖`가 왜 그 값이 되는지(규제 강도 `C`와의 관계)는
> 오늘 범위 밖임. 지금 확인한 건 "비례한다"까지임.

### 왜 Sex를 추가하면 분포가 확 벌어질까?

---

#### Pclass만 썼을 때의 한계

Pclass만 보면:
- 사망과 생존의 확률 분포가 **많이 겹쳐 있음**
- AUC가 0.5(랜덤)는 아니니까 Pclass는 분명 영향이 있음
- 하지만 3등석 중에도 생존자가 있고, 1등석 중에도 사망자가 있음
- → 예측 확률이 **딱 3개 값**만 나옴: `0.651`(1등) / `0.438`(2등) / `0.245`(3등)

**왜 3개뿐임?** 피처가 `Pclass` 하나고, 값이 1·2·3 세 가지뿐이니까.
모델이 만들 수 있는 확률도 3개가 전부임. 그래서 분포가 뭉텅이로 겹침.

> Pclass만 쓰면 모델이 이렇게 말하는 셈:  
> *"3등석이면 좀 위험하긴 한데… 100%는 아니야"*

**"분포가 겹친다"** = 어떤 구간에서 사망도 많고 생존도 많다 = threshold 0.5에서 헷갈린다!

---

#### Sex를 추가하면 왜 급변하는가?

타이타닉에는 거의 **규칙**처럼 작동한 게 있었음:

> **"Women and children first"** (여성과 아이 먼저)

> **주의**: 모델은 "규칙"을 배우는 게 아니라 **데이터에 남은 패턴**을 배움.
> 아래 숫자는 *역사적 규칙*이 아니라 *이 데이터에서 관찰된 비율*임.

![현실의 규칙이 수식의 가중치(Weight)로 변환되었습니다](스크린샷%202026-02-16%20오후%202.12.24.png)

| 그룹 | 생존률 (train.csv 실측) |
|------|--------|
| 여성 | **74.2%** |
| 남성 | **18.9%** |

이건 **엄청 강한 신호**!

> 위 슬라이드에 적힌 확률 범위(0.7~0.9 / 0.05~0.2)는 **개념 설명용 예시임.**
> 이 데이터의 실제 값은 아래에서 직접 뽑아서 확인함.


모델 관점에서 보면: `z = w₁·Pclass + w₂·Sex + b`

| Sex 값 | z 방향 | sigmoid 후 확률 |
|--------|--------|----------------|
| 여성 | z가 **양수** 쪽으로 밀림 | 높아짐 |
| 남성 | z가 **음수** 쪽으로 밀림 | 낮아짐 |

→ **양쪽으로 갈라짐!**

**그런데 얼마나 갈라짐?** 그건 `w₂`가 얼마나 큰지에 달렸음.
그 `w`를 **직접 꺼내서 볼 거임** (조금 뒤에).

---

#### AUC와의 연결 — 3-2회차 콜백

3-2회차에서 배운 **ROC-AUC**를 기억하셈.

AUC = "생존자가 사망자보다 더 높은 점수를 받을 확률"

Sex가 들어가면서:
- 거의 모든 여성(생존 많음) > 거의 모든 남성(사망 많음)
- → 순위 정렬이 훨씬 정확해짐 → **AUC 급상승**

> **한 줄 요약**: Sex가 추가되면서 모델이 생존과 사망을  
> 거의 **양쪽 극단으로 밀어버릴 수 있게** 되었기 때문!

---

#### 그렇다면 Sex 하나만 써도 AUC가 높을까?

Sex가 그만큼 강한 신호라면, **Sex 하나만으로도** AUC가 꽤 높을 수 있지 않을까?
아래 코드로 직접 확인해보기!

---

### [4-1] Prediction Card 7 — 아이

> **"Women and children first"의 뒷부분도 데이터에 보임?**

지금까지 확인한 건 **여성** 부분뿐임. `Sex` 하나만 봤음.

**아이**는 어땠을까? 실행 전에 예측하셈.

| 그룹 | 생존률 예상 |
|---|---|
| 남자 아이 (0~12세) | |
| 남자 성인 | |
| **여자 아이 (0~12세)** | |
| **여자 성인** | |

- 아이가 성인보다 무조건 높을 것 같음?
- **여자 아이**와 **여자 성인** 중 어느 쪽이 높을 것 같음?

In [17]:
age_df = titanic_df.dropna(subset=["Age"]).copy()
age_df["연령대"] = age_df["Age"].apply(lambda a: "아이(0~12)" if a <= 12 else "성인(13+)")

display(
    age_df.groupby(["Sex", "연령대"])["Survived"]
          .agg(인원="size", 생존률="mean")
          .round(3)
)


인원    생존률
Sex    연령대                 
female 성인(13+)   229  0.777
       아이(0~12)   32  0.594
male   성인(13+)   416  0.173
       아이(0~12)   37  0.568

### 결과 — 절반만 맞았음

- **남자**: 아이(0.568)가 성인(0.173)보다 **3배 이상** 높음 → "아이 먼저"가 강하게 작동
- **여자**: 아이(0.594)가 성인(0.777)보다 **오히려 낮음** → "아이 먼저"가 안 보임

> 즉 **"Women and children first"는 남성 안에서만 뚜렷하게 나타남.**
> 여성은 나이와 무관하게 이미 생존률이 높았음.

---

#### 그래서 뭐가 문제임?

우리 모델은 `Age`를 **숫자 하나로** 넣었음. 그러면 모델은
"나이가 1살 늘면 z가 얼마 변한다"는 **직선 관계**만 배울 수 있음.

그런데 방금 본 패턴은 **성별에 따라 나이의 효과가 다름**(상호작용)임.
지금 모델 구조로는 이걸 표현할 방법이 없음.

> **오늘의 질문**: 이 패턴을 모델에 알려주려면 어떻게 해야 할까?
> (힌트: `is_child` 같은 컬럼을 직접 만들어주면 어떻게 될까?)

> **확인한 만큼만 말한다**: 위 표는 `Age` 결측(177명)을 뺀 714명 기준임.
> 결측을 median으로 채운 모델과는 대상이 다름.

---

### [4-1] Prediction Card 8 — 계수

> **Sex의 계수는 Pclass보다 클까?**

방금 "Sex가 강한 신호"라고 했음. 그게 사실이면 **모델이 학습한 계수에 그대로 찍혀 있어야 함.**

실행 전에 두 개만 예측하셈.

1. `Sex`의 계수 크기 vs `Pclass`의 계수 크기 — 어느 쪽이 큼?
2. **3등석 여성**과 **1등석 남성** 중 누구의 생존 확률이 높게 나올 것 같음?

> 2번은 함정처럼 보이지만 함정 아님. 계수를 보면 답이 나옴.

In [18]:
import pandas as pd

m_ps = make_lr_pipeline(["Pclass", "Sex"])
m_ps.fit(X_train, y_train)

names = m_ps.named_steps["preprocess"].get_feature_names_out()
coefs = m_ps.named_steps["model"].coef_[0]
b = m_ps.named_steps["model"].intercept_[0]

display(pd.DataFrame({"피처": names, "계수 w": coefs.round(3)}))
print(f"절편 b: {b:.3f}")
print()

combo = X_valid[["Pclass", "Sex"]].copy()
combo["예측 확률(생존)"] = m_ps.predict_proba(X_valid)[:, 1].round(3)
display(
    combo.drop_duplicates()
         .sort_values(["Sex", "Pclass"])
         .reset_index(drop=True)
)


,피처,계수 w
0,num__Pclass,-0.826
1,cat__Sex_female,1.307
2,cat__Sex_male,-1.306


절편 b: -0.248



,Pclass,Sex,예측 확률(생존)
0,1,female,0.913
1,2,female,0.796
2,3,female,0.592
3,1,male,0.436
4,2,male,0.223
5,3,male,0.096


### 계수 해석 — 슬라이드가 말한 게 실제로 맞았음

슬라이드에서 이렇게 말했음:

> If Female → **Positive Weight (+)** → Score z High
> If Male → **Negative Weight (−)** → Score z Low

출력된 계수를 보셈. **부호가 실제로 그렇게 나옴.**
그리고 `Pclass`는 **음수** — 등급 숫자가 커질수록(3등석) 생존 확률이 내려감.

---

#### 확률이 6개밖에 안 나오는 이유

`Pclass`(3가지) × `Sex`(2가지) = **6조합**.
피처가 2개뿐이니 모델이 만들 수 있는 확률도 6개가 전부임.

여기서 아까 예측한 2번을 확인하셈.

- **3등석 여성** vs **1등석 남성** — 누가 높았음?
- 이 결과가 나온 이유를 계수 표로 설명할 수 있음?

> **한 줄**: 성별이 만드는 z의 차이가 등급 3단계가 만드는 차이보다 커서,
> **등급을 두 단계 낮춰도 성별을 뒤집지 못함.**

---

#### 이게 로지스틱 회귀의 진짜 장점임

- KNN은 "왜 그렇게 분류했는지" 물어보면 **답을 못 함** (거리만 계산함)
- 로지스틱은 **계수를 꺼내서 근거를 댈 수 있음**
- 4-2에서 KNN을 배울 때 이 차이를 다시 만날 거임

> **확인한 만큼만 말한다 — 계수 크기 비교는 조심해야 함**
>
> `Pclass`는 `StandardScaler`로 표준화된 값이고(1 = 1표준편차),
> `Sex_female`은 OneHot이라 0 또는 1임. **단위의 의미가 서로 다름.**
> 그래서 `|1.221| > |0.802|` 라는 것만으로 "Sex가 더 중요하다"고 단정하면 안 됨.
>
> 대신 **위 확률 표를 보면 결론이 남**: **3등석 여성**이 **1등석 남성**보다 높음.
> 등급을 두 단계 낮춰도 성별을 못 뒤집었음. **이건 계수 해석이 아니라 실측임.**

In [19]:
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

configs_ext = [
    ("Pclass만", ["Pclass"]),
    ("Sex만", ["Sex"]),
    ("Pclass + Sex", ["Pclass", "Sex"]),
    ("전체 피처", X_train.columns.tolist()),
]

rows = []
for name, feats in configs_ext:
    m = make_lr_pipeline(feats)
    m.fit(X_train, y_train)
    p = m.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, p)
    acc = accuracy_score(y_valid, (p >= 0.5).astype(int))
    rows.append([name, len(feats), f"{auc:.3f}", f"{acc:.3f}"])

display(pd.DataFrame(rows, columns=["모델", "원본 컬럼 수", "AUC", "Accuracy"]))

,모델,원본 컬럼 수,AUC,Accuracy
0,Pclass만,1,0.702,0.669
1,Sex만,1,0.780,0.798
2,Pclass + Sex,2,0.838,0.798
3,전체 피처,8,0.857,0.798


#### 해석

- **Sex 하나만으로도** Pclass보다 높은 AUC를 보임
- Sex + Pclass 두 개만 합쳐도 전체 피처에 근접하는 성능
- 나머지 피처(Age, Fare 등)는 **예외 케이스**를 더 잘 분리해주는 역할

| 피처 | 역할 |
|------|------|
| **Sex** | 가장 강한 신호 — 생존/사망의 **큰 틀**을 결정 |
| **Pclass** | 보조 신호 — 같은 성별 안에서 **추가 구분** |
| **나머지** | 미세 조정 — **예외 케이스** 처리 |

> **교훈**: 피처를 많이 넣는다고 무조건 좋은 게 아니라,  
> **강한 신호 1~2개**가 성능의 대부분을 결정할 수 있다!

---

#### 잠깐 — "원본 컬럼 수 8"의 함정

표의 `원본 컬럼 수`는 **인코딩 전 컬럼 개수**임.
Sex/Embarked/Cabin1을 OneHot으로 펼치면 모델이 실제로 보는 입력은 8개보다 훨씬 많음.

```python
# 직접 세어보셈
m = make_lr_pipeline(X_train.columns.tolist())
m.fit(X_train, y_train)
m.named_steps["preprocess"].transform(X_train).shape
```

---

#### 3회차 콜백 — 인코딩을 바꾸면 성능이 달라질까?

이 노트북은 3회차에서 배운 **OneHotEncoder**로 범주형을 처리함.
예전 방식(범주형을 그냥 정수 라벨로 바꾸기)과 비교하면 얼마나 차이 날까?

**Prediction — 먼저 예상해보셈**: OneHot이 더 좋을 것 같음? 얼마나?


In [20]:
from sklearn.preprocessing import OrdinalEncoder

def make_pipeline_with(encoder_kind):
    num = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
    cat = ["Sex", "Embarked", "Cabin1"]
    enc = (OneHotEncoder(handle_unknown="ignore") if encoder_kind == "onehot"
           else OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    pre = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("enc", enc)]), cat),
    ], remainder="drop")
    return Pipeline([("preprocess", pre),
                     ("model", LogisticRegression(max_iter=5000, random_state=42))])

rows = []
for kind, label in [("ordinal", "정수 라벨 (예전 방식)"), ("onehot", "OneHot (지금 방식)")]:
    mm = make_pipeline_with(kind).fit(X_train, y_train)
    pp = mm.predict_proba(X_valid)[:, 1]
    rows.append([label,
                 mm.named_steps["preprocess"].transform(X_train).shape[1],
                 round(roc_auc_score(y_valid, pp), 4),
                 round(accuracy_score(y_valid, (pp >= 0.5).astype(int)), 4)])

display(pd.DataFrame(rows, columns=["인코딩 방식", "모델이 보는 차원", "AUC", "Accuracy"]))


,인코딩 방식,모델이 보는 차원,AUC,Accuracy
0,정수 라벨 (예전 방식),8,0.8566,0.7809
1,OneHot (지금 방식),19,0.8570,0.7978


### 해석 — 차이가 생각보다 안 남

**AUC는 사실상 같고, Accuracy만 조금 다름.**

"OneHot이 정답이니까 성능이 확 오르겠지"라고 기대했다면 빗나갔을 거임.

---

#### 그럼 OneHot을 왜 씀?

**성능 때문이 아님. 의미 때문임.**

정수 라벨은 `Embarked: C=0, Q=1, S=2` 처럼 **없던 순서를 만들어냄.**
모델은 "S가 C보다 2만큼 크다"로 읽음. 그런 순서는 원래 없었음.

> 3회차에서 배운 그것임 — **도구를 제자리에 안 쓰면 없던 순서가 전달됨.**

- 이번 데이터에서는 **우연히** 그 잘못된 순서가 성능에 큰 해를 안 끼쳤음
- 하지만 **범주가 많아지거나 순서가 성능에 영향을 주는 데이터면 얘기가 달라짐**

> **오늘의 문장**: 전처리를 제대로 하는 이유는 **점수가 오르기 때문이 아니라, 모델이 데이터를 오해하지 않게 하기 위해서**임.

> **확인한 만큼만 말한다**: 위 표는 이 valid 178명 기준임.
> split을 바꾸면 두 방식의 순위가 뒤바뀔 수도 있음. **차이가 작다는 게 요점임.**

---

## Part 6. 마지막 — 수능 보러 가기

지금까지 우리가 한 일을 정리하면:

| 무엇을 골랐나 | 어디서 골랐나 |
|---|---|
| 전처리 방식 (OneHot + Scaler) | train에서 `fit` |
| 임곗값 | **valid** 표를 보고 |
| 피처 조합 | **valid** AUC를 보고 |

**전부 valid를 보고 고름.** 그래서 valid 점수는 이미 **"내가 고른 답이 잘 맞는지"** 를 잰 것이지,
**"처음 보는 사람에게 통하는지"** 를 잰 게 아님.

`X_final_test`은 노트북 시작부터 지금까지 **한 번도 안 나왔음.** 이제 딱 한 번 씀.

---

### [4-1] Prediction Card 9 — 최종 시험

바로 앞 표에서 나온 valid 최종 성적(전체 피처)을 확인하고 오셈.

| | test 예상 |
|---|---|
| Accuracy | |
| AUC | |

- 더 높게 나올까, 낮게 나올까, 비슷할까?
- **왜** 그렇게 생각함?

In [21]:
from sklearn.metrics import roc_auc_score, accuracy_score

# valid에서 고른 것들을 그대로 고정한 뒤, test은 딱 한 번만 평가함
final_model = make_lr_pipeline(X_train.columns.tolist())
final_model.fit(X_train, y_train)

p_valid = final_model.predict_proba(X_valid)[:, 1]
p_final_test = final_model.predict_proba(X_final_test)[:, 1]

rows = [
    ["valid  (모의고사)", len(y_valid),
     f"{roc_auc_score(y_valid, p_valid):.3f}",
     f"{accuracy_score(y_valid, (p_valid >= 0.5).astype(int)):.3f}"],
    ["test (수능)", len(y_final_test),
     f"{roc_auc_score(y_final_test, p_final_test):.3f}",
     f"{accuracy_score(y_final_test, (p_final_test >= 0.5).astype(int)):.3f}"],
]
display(pd.DataFrame(rows, columns=["세트", "인원", "AUC", "Accuracy"]))


,세트,인원,AUC,Accuracy
0,valid (모의고사),178,0.857,0.798
1,test (수능),179,0.829,0.782


### 결과 해석 — 이 차이가 오늘의 결론임

두 숫자가 **다르게 나왔을 것임.** 그게 정상임.

- valid 점수는 **내가 그걸 보고 골랐기 때문에** 조금 유리하게 나옴
- test 점수는 **아무것도 안 보고 잰 것**이라 더 정직함

> **어느 쪽을 "이 모델의 성능"이라고 보고해야 함?**
> **test 쪽임.** valid 점수는 "고르는 데 쓴 점수"지 "성능"이 아님.

---

#### 만약 3분할을 안 했다면?

임곗값도 test에서 고르고, 피처도 test에서 고르고, 그 test 점수를 성능이라고 발표했을 거임.
그건 **답안지를 보고 푼 시험 점수**임.

> **3회차 → 4회차로 이어진 원칙**
>
> | 회차 | 무엇까지 넓혔나 |
> |---|---|
> | 2회차 | **모델**을 valid로 고름 |
> | 3회차 | **전처리**까지 Pipeline 안으로 (누수 방지) |
> | **4회차** | **임곗값·피처**까지 valid로 고름 |
>
> 새 규칙을 외운 게 아님. **같은 규칙이 계속 넓어진 것뿐임.**

> **확인한 만큼만 말한다**: test이 valid보다 높게 나올 수도 있음.
> 179명짜리 표본이라 흔들림이 큼. **방향이 아니라 "따로 재야 한다"가 요점임.**

---
## 오늘의 정리

| 개념 | 핵심 | 코드 |
|------|------|------|
| 분류 | 데이터에 라벨 붙이기 | — |
| Sigmoid | 점수(z) → 확률(0~1) 변환기 | `1 / (1 + exp(-z))` |
| 로지스틱 회귀 | 확률 기반 분류기 | `LogisticRegression()` |
| predict_proba | 각 클래스의 확률 출력 | `lr.predict_proba(X)[:,1]` |
| 임곗값 | 확률 → 예측 변환 기준 | `(y_proba >= thr).astype(int)` |
| Precision/Recall | 임곗값에 따라 줄다리기 | 정책에 맞게 조정 |

> **기억할 흐름**:  
> 로지스틱 회귀 = **Sigmoid 함수** → 점수(z)를 **확률로 변환** → **임곗값에 따라 분류** → **정책에 맞게 임곗값을 조정**

---
## 다음: 4-2 KNN

- **KNN**: 거리 기반 다수결 분류기
- **K 값**에 따른 경계 변화 (과적합 ↔ 과소적합)
- **스케일링**이 왜 필수인지
- **Pipeline + GridSearchCV**로 최적 K 탐색
- **Decision Boundary** 시각화

> 로지스틱 회귀는 **확률**로 분류했다면,  
> KNN은 **가까운 이웃들의 다수결**로 분류!